# Mercadona API client

An alternative to `scrapers/mercadona.ipynb` (Selenium), kept as an option to migrate
to. Instead of typing searches into the website, we query the JSON API that
`tienda.mercadona.es` itself uses:

| Step | Endpoint |
|---|---|
| Postal code → warehouse | `PUT /api/postal-codes/actions/change-pc/` (warehouse returned in the `x-customer-wh` header) |
| Category tree | `GET /api/categories/?lang=es&wh=...` |
| Products of a category | `GET /api/categories/{id}/?lang=es&wh=...` |

Mercadona's API has **no search endpoint** (the website's search runs on a third-party
service, Algolia). So we download the **full catalogue** category by category
(~150 requests) and then assign each product to one of the project's 20 search terms
by **keyword matching** on its name.

Differences from the Selenium scraper:

- Every product has a stable `product_id`, a `url` and its Mercadona `category`.
- Unit price and pack size come structured from the API instead of being parsed from text.
- `searched_item` is a keyword match, not the website's search ranking: it avoids
  unrelated results (e.g. coffee when searching "leche") but can miss products whose
  name does not contain the term.
- The warehouse is resolved from the postal code instead of being hard-coded
  (28911 resolves to `mad1`).

### Ethical note (robots.txt)

Mercadona's `robots.txt` includes `Disallow: /api`. This is a request, not a technical
block or a legal rule, but we state it openly. To minimise impact: academic,
non-commercial use, ~150 requests per full run with a 1 s pause between requests, and
a `User-Agent` that identifies the project instead of impersonating a browser.

### Outputs

- `data/raw/mercadona_api_products_raw.csv`: the full catalogue, one row per product.
- `data/processed/mercadona_api_products.csv`: only products matching a search term,
  with the same columns as `mercadona_products.csv` plus `product_id`, `url`,
  `category` and `previous_price`.

In [ ]:
import re
import time
import unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

BASE_URL = "https://tienda.mercadona.es"
POSTAL_CODE = "28911"
LANG = "es"

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------
TEST_MODE = False
TEST_MAX_CATEGORIES = 5   # in test mode, only the first N subcategories

REQUEST_PAUSE = 1.0       # seconds between requests
MAX_RETRIES = 3
TIMEOUT = 20

HEADERS = {
    "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0 (academic, non-commercial)",
    "Accept": "application/json",
}

In [ ]:
current = Path.cwd().resolve()
BASE_DIR = current.parent if current.name == "api_clients" else current

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "mercadona_api_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "mercadona_api_products.csv"

print("RAW:", RAW_PATH)
print("PROCESSED:", PROCESSED_PATH)

## API requests

`request_json` makes one request with retries (exponential backoff on network errors,
429 or 5xx) and pauses after every call. `resolve_warehouse` asks the API which
warehouse serves our postal code, since prices and stock are tied to it.

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)


def request_json(method, path, **kwargs):
    url = BASE_URL + path

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.request(method, url, timeout=TIMEOUT, **kwargs)

            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}")

            response.raise_for_status()
            time.sleep(REQUEST_PAUSE)
            return response

        except requests.RequestException as exc:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({exc}). Retrying in {wait} s...")
            time.sleep(wait)


def resolve_warehouse(postal_code):
    response = request_json(
        "PUT",
        "/api/postal-codes/actions/change-pc/",
        json={"new_postal_code": postal_code},
    )
    warehouse = response.headers.get("x-customer-wh")
    if not warehouse:
        raise RuntimeError("The API did not return a warehouse for this postal code.")
    return warehouse


WAREHOUSE = resolve_warehouse(POSTAL_CODE)
print(f"Postal code {POSTAL_CODE} -> warehouse {WAREHOUSE}")

## Category tree

The tree has two levels: ~26 top-level categories (e.g. *Aceite, especias y salsas*)
containing ~150 subcategories (e.g. *Aceite de oliva*). Products are only listed at
subcategory level, grouped into sections.

In [ ]:
def get_subcategories():
    data = request_json(
        "GET", "/api/categories/", params={"lang": LANG, "wh": WAREHOUSE}
    ).json()

    subcategories = []
    for top in data["results"]:
        for sub in top["categories"]:
            subcategories.append({
                "top_category": top["name"],
                "subcategory_id": sub["id"],
                "subcategory": sub["name"],
            })
    return subcategories


subcategories = get_subcategories()

if TEST_MODE:
    subcategories = subcategories[:TEST_MAX_CATEGORIES]

print("Subcategories to fetch:", len(subcategories))

## Flatten the JSON

The useful fields are in `price_instructions`:

- `unit_price`: price of the item as sold (one bottle, one pack...).
- `reference_price` + `reference_format`: price per reference unit (`kg`, `L`,
  `100 g`, `100 ml`, `ud`, `dc`/`dz` = dozen). This is the price-per-unit the
  website shows.
- `unit_size` + `size_format`: total quantity of the item (`kg`, `l` or `ud`).
- `is_pack`, `total_units`, `pack_size`, `unit_name`: pack composition
  (e.g. 12 mini bottles x 0.1 kg).
- `previous_unit_price`: the price before a recent price change, if any.

In [ ]:
def describe_quantity(pi):
    """Builds a readable quantity string, e.g. '12 mini botellas x 0.1 kg' or '1.5 l'."""
    size_format = pi.get("size_format") or ""

    if pi.get("is_pack") and pi.get("total_units") and pi.get("pack_size"):
        unit_name = pi.get("unit_name") or "ud."
        return f"{pi['total_units']} {unit_name} x {pi['pack_size']} {size_format}"

    if pi.get("unit_size") is not None:
        return f"{pi['unit_size']} {size_format}"

    return None


def flatten_product(product, subcategory, section, scraped_at):
    pi = product.get("price_instructions") or {}

    return {
        "supermarket": "Mercadona",
        "product_id": product.get("id"),
        "name": product.get("display_name"),
        "packaging": product.get("packaging"),
        "top_category": subcategory["top_category"],
        "category": subcategory["subcategory"],
        "section": section,
        "unit_price": pi.get("unit_price"),
        "previous_unit_price": pi.get("previous_unit_price"),
        "price_decreased": pi.get("price_decreased"),
        "reference_price": pi.get("reference_price"),
        "reference_format": pi.get("reference_format"),
        "bulk_price": pi.get("bulk_price"),
        "unit_size": pi.get("unit_size"),
        "size_format": pi.get("size_format"),
        "is_pack": pi.get("is_pack"),
        "total_units": pi.get("total_units"),
        "pack_size": pi.get("pack_size"),
        "unit_name": pi.get("unit_name"),
        "approx_size": pi.get("approx_size"),
        "quantity_raw": describe_quantity(pi),
        "url": product.get("share_url"),
        "image_url": product.get("thumbnail"),
        "postal_code": POSTAL_CODE,
        "warehouse": WAREHOUSE,
        "scraped_at": scraped_at,
    }

## Download the catalogue

If a subcategory still fails after the retries, it is logged as `ERROR` in the
summary and the loop moves on to the next one.

In [ ]:
rows = []
summary = []

for i, subcategory in enumerate(subcategories, start=1):
    print(f"[{i}/{len(subcategories)}] {subcategory['top_category']} > {subcategory['subcategory']}")
    started = time.monotonic()

    try:
        data = request_json(
            "GET",
            f"/api/categories/{subcategory['subcategory_id']}/",
            params={"lang": LANG, "wh": WAREHOUSE},
        ).json()

        scraped_at = datetime.now().isoformat(timespec="seconds")
        count = 0

        for section in data.get("categories", []):
            for product in section.get("products", []):
                rows.append(flatten_product(product, subcategory, section["name"], scraped_at))
                count += 1

        status = "OK"

    except Exception as exc:
        print(f"  ERROR: {type(exc).__name__}: {exc}")
        count, status = 0, "ERROR"

    summary.append({
        **subcategory,
        "products": count,
        "status": status,
        "seconds": round(time.monotonic() - started, 2),
    })

df_summary = pd.DataFrame(summary)
print(df_summary["status"].value_counts())
display(df_summary[df_summary["status"] != "OK"])

In [ ]:
# --------------------------------------------------
# SAVE RAW CSV
# --------------------------------------------------
df_raw = pd.DataFrame(rows)

df_raw.to_csv(RAW_PATH, index=False, encoding="utf-8-sig")

print("Raw rows:", len(df_raw))
print("Distinct product IDs:", df_raw["product_id"].nunique())
print("RAW saved:", RAW_PATH)

## Assign search terms

Names and terms are lowercased and stripped of accents, then each term is matched as
a **whole word**, allowing a plural ending (`manzana` matches *Manzanas*, `pan`
does not match *Panceta*). A product matching several terms keeps the first one in
`SEARCH_TERMS` order, and all matches are kept in `matched_terms`.

Known limitation: whole-word matching still catches some unrelated products
(e.g. `leche` matches *Chocolate con leche*), just like a website search would.

In [ ]:
def strip_accents(text):
    normalized = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in normalized if not unicodedata.combining(c))


TERM_PATTERNS = {
    term: re.compile(r"\b" + re.escape(strip_accents(term)) + r"(?:s|es)?\b")
    for term in SEARCH_TERMS
}


def match_terms(name):
    clean = strip_accents(name)
    return [term for term, pattern in TERM_PATTERNS.items() if pattern.search(clean)]


df_raw["matched_terms"] = df_raw["name"].map(match_terms)
df_raw["searched_item"] = df_raw["matched_terms"].map(lambda terms: terms[0] if terms else None)
df_raw["matched_terms"] = df_raw["matched_terms"].map("|".join)

print("Products matching a term:", df_raw["searched_item"].notna().sum(), "of", len(df_raw))
print(df_raw["searched_item"].value_counts())

## Processing

1. **Price per unit**: `reference_price` is converted to the project's common units.
   A `100 g` / `100 ml` reference is multiplied by 10 to give €/kg or €/L; `dc`/`dz`
   (dozen) becomes `DOCENA` and `lv` (wash) becomes `LAVADO`, like in DIA.
2. **Total quantity**: `unit_size` is already expressed in `kg`, `l` or `ud`.
3. **Deduplication** by `product_id` (a product can appear in two subcategories).
4. Only products matching a search term are kept, so the file can replace
   `mercadona_products.csv`.

In [ ]:
UNIT_MAP = {
    # API unit: (project unit, factor to base unit)
    "kg": ("KILO", 1), "g": ("KILO", 0.001),
    "l": ("LITRO", 1), "ml": ("LITRO", 0.001),
    "ud": ("UNIDAD", 1),
    "dc": ("DOCENA", 1), "dz": ("DOCENA", 1),
    "lv": ("LAVADO", 1),
}

SIZE_UNITS = {"kg": "KILO", "l": "LITRO", "ud": "UNIDAD"}


def normalize_unit_price(reference_price, reference_format):
    """'kg' -> (price, KILO); '100 g' -> (price * 10, KILO); 'dc' -> (price, DOCENA)."""
    if pd.isna(reference_price) or pd.isna(reference_format):
        return None, None

    match = re.match(r"^\s*(\d+(?:[.,]\d+)?)?\s*([a-z]+)\s*$", str(reference_format).lower())
    if not match:
        return None, None

    amount = float(match.group(1).replace(",", ".")) if match.group(1) else 1.0
    mapped = UNIT_MAP.get(match.group(2))
    if not mapped or amount == 0:
        return None, None

    unit, factor = mapped
    return round(float(reference_price) / (amount * factor), 4), unit


def process(df):
    df = df[df["searched_item"].notna()].copy()
    df = df.drop_duplicates(subset=["product_id"]).reset_index(drop=True)

    df["price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df["price_raw"] = df["unit_price"].astype(str) + " €"
    df["price_type"] = df["is_pack"].map({True: "/pack", False: "/ud"})
    df["previous_price"] = pd.to_numeric(df["previous_unit_price"], errors="coerce")

    normalized = [
        normalize_unit_price(p, f)
        for p, f in zip(pd.to_numeric(df["reference_price"], errors="coerce"),
                        df["reference_format"])
    ]
    df["price_per_unit"] = [n[0] for n in normalized]
    df["unit"] = [n[1] for n in normalized]

    df["total_quantity"] = df["unit_size"]
    df["total_quantity_unit"] = df["size_format"].map(SIZE_UNITS)

    columns = [
        # same columns as mercadona_products.csv
        "searched_item", "supermarket", "name", "quantity_raw", "price_raw",
        "price_type", "price", "image_url", "scraped_at", "total_quantity",
        "unit", "price_per_unit", "postal_code", "warehouse",
        # extra columns from the API
        "product_id", "url", "category", "previous_price", "matched_terms",
        "total_quantity_unit",
    ]
    return df[columns]

In [ ]:
df_processed = process(df_raw)
before_drop = len(df_processed)

unknown_formats = df_raw.loc[
    df_raw["product_id"].isin(df_processed.loc[df_processed["unit"].isna(), "product_id"]),
    "reference_format"
].value_counts(dropna=False)

df_processed = df_processed[
    df_processed["price"].notna()
    & df_processed["price_per_unit"].notna()
    & df_processed["unit"].notna()
].reset_index(drop=True)

df_processed.to_csv(PROCESSED_PATH, index=False, encoding="utf-8-sig")

print("Unique matched products:", before_drop)
print("Final products:", len(df_processed))
print("Dropped for missing price/unit:", before_drop - len(df_processed))
if not unknown_formats.empty:
    print("Unrecognized reference formats:")
    print(unknown_formats)
print("\nUnits:")
print(df_processed["unit"].value_counts(dropna=False))
print("PROCESSED saved:", PROCESSED_PATH)